<a href="https://colab.research.google.com/github/feixh/colab-notebooks/blob/main/position_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Position Embeddings

This notebook experiments with absolute position embedding, 1D RoPE (rotational position embedding), and 2D RoPE which is often used in vision tasks.


**Compute requirement**: CPU instance

In [ ]:
# Install the dependencies
# We use jaxtyping (https://github.com/patrick-kidger/jaxtyping) for annotating types of tensors,
# and einops (https://github.com/arogozhnikov/einops) for easy manipulation of tensors

!pip install jaxtyping einops

## Absolute position embedding

The absolute position embedding of the token at index $t$ (where $t$ is an integer no less than zero) is a $d$-dimensional vector $x \in \mathbb{R}^d$, the $i$-th and $(i+1)$-th component of the feature is a 2-tuple $(\cos(\theta_i), \sin(\theta_i))$ where $i=0,1,\cdots, \frac{d}{2}$ assuming $d$ is even.

So what's $\theta_i$?

$$
\theta_i = t / 10000^\frac{i}{d/2}
$$

If we define the frequency $f_i := 1 / 10000^\frac{i}{d/2}$ (which ranges from 1 to 1 / 10000), then $\theta_i = t \cdot f_i$, and the corresponding wavelength $\omega_i = 2 \pi / f_i$ ranges from $2\pi$ to $10000\cdot 2\pi$.


The main reference is the famous [*Attention is All You Need*](https://arxiv.org/abs/1706.03762) paper that led the revolution of large language models (LLMs).

In [ ]:
from jaxtyping import Float
import torch
from torch import Tensor
from torch import nn
import math
from einops import einsum


def absolute_position_encoding(seq_len: int, dim: int, base: int = 10_000) -> Float[Tensor, "D"]:
  assert dim % 2 == 0
  half_dim = dim // 2
  freq = torch.exp(-math.log(base) * torch.arange(half_dim, dtype=torch.float32) / half_dim)
  th = einsum(torch.arange(seq_len, dtype=torch.float32), freq, "S, D -> S D")
  cos_th = torch.cos(th)  # [S, D//2]
  sin_th = torch.sin(th)  # [S, D//2]
  feat = torch.zeros(seq_len, dim)
  feat[:, ::2] = cos_th
  feat[:, 1::2] = sin_th
  return feat


def test_absolute_position_encoding():
  import matplotlib.pyplot as plt
  s = 256
  dim = 512
  feat = absolute_position_encoding(s, dim) # (D)
  plt.imshow(feat)
  plt.title("absolute position embedding")
  plt.xlabel("each row of the image is the position embedding of the corresponding token index")
  plt.ylabel("token index")
  plt.show()

test_absolute_position_encoding()


## RoPE (Rotational Position Embedding)

For RoPE, given the $d$-dimensional feature embedding $x \in \mathbb{R}^d$ of the token at index $t$, the output feature $y \in \mathbb{R}^d$ has the property that each 2-tuple elements of $y$:
$$
\begin{bmatrix}
y_i \\
y_{i+1}
\end{bmatrix}
=
\begin{bmatrix}
\cos(\theta_i) & -\sin(\theta_i) \\
\sin(\theta_i) & \cos(\theta_i)
\end{bmatrix}
\begin{bmatrix}
x_i \\
x_{i+1}
\end{bmatrix}
$$
where $\theta_i = t / 10000^\frac{i}{d/2}$.

Note the matrix
$$
\begin{bmatrix}
\cos(\theta_i) & -\sin(\theta_i) \\
\sin(\theta_i) & \cos(\theta_i)
\end{bmatrix}
$$
is a 2-D rotation matrix that rotates its right hand side -- a 2-D vector $[x_i, x_{i+1}]^\top$ -- by the amount of $\theta_i$ counter-clockwise to its left hand side $[y_i, y_{i+1}]^\top$.

In an actual implementation, to simplify the manipulation of the indices, one can rotate the pair $(i, i + d/2)$ instead of $(i, i+1$):
$$
\begin{bmatrix}
y_i \\
y_{i+d/2}
\end{bmatrix}
=
\begin{bmatrix}
\cos(\theta_i) & -\sin(\theta_i) \\
\sin(\theta_i) & \cos(\theta_i)
\end{bmatrix}
\begin{bmatrix}
x_i \\
x_{i+d/2}
\end{bmatrix}
$$

The key point here is that one rotates pairs of elements of each token's embedding to produce the output.

Note, the rotation matrices applied to pairs of values are parametrized by the token index $t$ and the dimension of the embedding $d$, and as such, we use the notation $\mathrm{RoPE}( ;t, d): \mathbb{R}^d \mapsto \mathbb{R}^d$ to represent the transformation and $y = \mathrm{RoPE}(x; t, d)$.



In [ ]:
class RoPE(nn.Module):
  def __init__(self, seq_len: int, dim: int, base: int = 10_000):
    super().__init__()
    assert dim % 2 == 0
    half_dim = dim // 2
    self.half_dim = half_dim
    freq = torch.exp(-math.log(base) * torch.arange(half_dim, dtype=torch.float32) / half_dim) # (D)
    idx = torch.arange(seq_len, dtype=torch.float32) # (S)
    th = einsum(idx, freq, "S, D -> S D")
    self.register_buffer("theta", th)
    self.register_buffer("cos_th", torch.cos(th)) # [S, D//2]
    self.register_buffer("sin_th", torch.sin(th)) # [S, D//2]

  def forward(self, x: Float[Tensor, "B S D"]) -> Float[Tensor, "B S D"]:
    cos_1st_half = einsum(self.cos_th, x[..., :self.half_dim], "S D, B S D -> B S D")  # [B, S, D//2]
    sin_1st_half = einsum(self.sin_th, x[..., :self.half_dim], "S D, B S D -> B S D")  # [B, S, D//2]
    cos_2nd_half = einsum(self.cos_th, x[..., self.half_dim:], "S D, B S D -> B S D")  # [B, S, D//2]
    sin_2nd_half = einsum(self.sin_th, x[..., self.half_dim:], "S D, B S D -> B S D")  # [B, S, D//2]
    y = torch.zeros_like(x)
    y[..., :self.half_dim] = cos_1st_half - sin_2nd_half
    y[..., self.half_dim:] = sin_1st_half + cos_2nd_half
    return y


def test_rope():
  bs = 8
  seq_len = 512
  dim = 256
  rope = RoPE(seq_len, dim)
  x = torch.randn((bs, seq_len, dim))
  y = rope(x)

test_rope()


## 2D RoPE


For a 2D $H\times W$ grid of tokens, the $d$-dim embedding of the token at grid index $(u, v)$ is $x \in \mathbb{R}^d$. We can apply the usual 1D RoPE to the first half of $x$ using index $u$, and the second half using index $v$. Using the notation introduced above for 1D RoPE, we have the following equation to transform the input $x$ to output $y$:

$$
\begin{aligned}
y[:d/2] = \mathrm{RoPE}(x[:d/2] ;u, d/2) \\
y[d/2:] = \mathrm{RoPE}(x[d/2:] ;v, d/2) \\
\end{aligned},
$$

that is, we apply 1D RoPE to each half of the embedding separately.



In [ ]:
from einops import rearrange

class RoPE2D(nn.Module):
  def __init__(self, h: int, w: int, dim: int, base: int = 10_000):
    super().__init__()
    assert dim % 4 == 0, "Dimension must be a multiple of 4 for 2D RoPE, since each spatial dimension gets half the features and requires an even number of features."
    self.h = h
    self.w = w
    self.dim = dim
    self.half_dim = dim // 2
    self.rope_h = RoPE(seq_len=h, dim=self.half_dim, base=base)
    self.rope_w = RoPE(seq_len=w, dim=self.half_dim, base=base)

  def forward(self, x: Float[Tensor, "B H W D"]) -> Float[Tensor, "B H W D"]:
    b, h, w, d = x.shape
    assert (h, w, d) == (self.h, self.w, self.dim)

    y_h = rearrange(
        self.rope_h(rearrange(x[..., :self.half_dim], "B H W D -> (B W) H D")),
        "(B W) H D -> B H W D", B=b)

    y_w = rearrange(
        self.rope_w(rearrange(x[..., self.half_dim:], "B H W D -> (B H) W D")),
        "(B H) W D -> B H W D", B=b)

    y = torch.cat((y_h, y_w), dim=-1)
    return y


def test_rope2d():
  bs = 8
  h = 256
  w = 512
  dim = 128
  rope2d = RoPE2D(h, w, dim)
  x = torch.randn((bs, h, w, dim))
  y = rope2d(x)

test_rope2d()
